# White-noise-robust separators for the tripartite cycle polytope

## Purpose

This notebook mirrors `Cycle_First_Stage_Clean.ipynb`, but replaces its
coefficient normalization by the white-noise normalization proposed for the
next LP test.

For each of the nonsignalling extremal classes $27$, $29$, $43$, and $44$, it
solves

$$
\begin{aligned}
\operatorname*{minimize}_{c,\beta}\quad
    &c\cdot p_{\mathrm{NS}}-\beta\\
\text{subject to}\quad
    &c\cdot v\geq\beta
      \qquad\forall v\in\operatorname{vert}(\mathcal C_{\mathrm{cycle}}),\\
    &c\cdot u-\beta=1,
\end{aligned}
$$

where $u(abc\mid xyz)=1/8$. Both the 64 coefficients $c_j$ and the bound
$\beta$ are free variables.

The vertex constraints make $c\cdot p\geq\beta$ valid on the complete
siglocal polytope. The last equality fixes the arbitrary positive scale by
giving the uniform behavior unit slack. Consequently, minimizing the signed
slack of the NS target maximizes its resistance to white noise.

The notebook is self-contained and performs the same structural audits as the
clean first-stage notebook before solving any LP.


## How to run

1. Open the notebook with the Julia kernel.
2. Choose **Kernel → Restart Kernel and Run All Cells**.
3. The first code cell installs only missing packages.
4. The four LPs should finish quickly.

The final cells display both the normalized rational inequalities and
primitive integer representatives. Machine-readable TSV files are written to
`cycle_white_noise_lp_results` beside the notebook.


## 1. Canonical probability coordinates

A behavior in the binary-input, binary-output tripartite scenario consists of

$$
p(abc\mid xyz),
\qquad a,b,c,x,y,z\in\{0,1\}.
$$

It is stored as a vector of length $2^6=64$. To remove any ambiguity, the vector is split into eight input blocks, ordered lexicographically as

$$
xyz=000,001,010,011,100,101,110,111.
$$

Inside each input block, the eight outputs use the same order,

$$
abc=000,001,010,011,100,101,110,111.
$$

Thus $z$ is the fastest-changing input bit and $c$ is the fastest-changing output bit. In Julia's one-based indexing, the coordinate of $p(abc\mid xyz)$ is

$$
1+c+2b+4a+8z+16y+32x.
$$

The binary outputs are also interpreted as signs when correlators are used later: output `0` corresponds to $+1$, and output `1` corresponds to $-1$.

The two assertions at the end of the next cell are small but important regression tests. They verify both the within-block output order and the order of the eight input blocks. Keeping this map explicit prevents the kind of silent iterator-order error that can change a polytope while leaving array dimensions apparently correct.


In [4]:
# Package bootstrap. Existing packages are not changed; only missing packages
# are installed in the active Julia environment.
import Pkg

required_packages = [
    "JuMP",
    "MathOptInterface",
    "GLPK",
]

missing_packages = [
    package for package in required_packages
    if Base.find_package(package) === nothing
]

if !isempty(missing_packages)
    println("Installing missing Julia packages: ", join(missing_packages, ", "))
    Pkg.add(missing_packages)
else
    println("All required Julia packages are already available.")
end


All required Julia packages are already available.


In [5]:
using LinearAlgebra
using Printf
using JuMP
using GLPK

# Import MathOptInterface directly. In particular, do not use
# `JuMP.MathOptInterface`, which caused an error with the user's Julia 1.12
# installation.
import MathOptInterface as MOI

const CLASSES = (27, 29, 43, 44)
const TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

#
# The 64-vector is ordered in blocks of fixed (x,y,z):
#
#   (x,y,z) = 000, 001, 010, 011, 100, 101, 110, 111.
#
# Within each block, the output order is
#
#   (a,b,c) = 000, 001, 010, 011, 100, 101, 110, 111.
#
# Thus c and z are the fastest-changing indices. Binary output 0 corresponds
# to the physicists' value +1, and binary output 1 corresponds to -1.

prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)


## 2. Correct deterministic vertices of the two cycle orientations

Each party uses a deterministic binary response function of two binary inputs. There are

$$
2^{2^2}=16
$$

such functions. The integer masks `0:15` encode their four-entry truth tables; `response(mask,u,v)` reads the output assigned to the pair $(u,v)$.

The two communication orientations have different causal dependencies:

| orientation | Alice | Bob | Charlie |
|---|---|---|---|
| clockwise | $a=f_A(x,z)$ | $b=f_B(y,x)$ | $c=f_C(z,y)$ |
| counterclockwise | $a=f_A(x,y)$ | $b=f_B(y,z)$ | $c=f_C(z,x)$ |

Choosing one of 16 response functions independently for each party gives $16^3=4096$ deterministic behaviors per orientation. Every deterministic behavior has exactly one probability equal to 1 in each input block and all other entries equal to 0.

The polytope used by the LP is the convex hull of the **union** of the two orientation lists. Their intersection contains the 64 fully local deterministic behaviors, so removing duplicate rows gives

$$
4096+4096-64=8128
$$

distinct vertices. Removing duplicates does not change the convex hull or any LP optimum, but it exposes the correct combinatorics and avoids redundant constraints.

This cell performs structural checks before any inequality is computed: binary entries, one selected output per input, expected vertex counts, intersection size, and affine dimensions. The individual orientation polytopes have affine dimension 44; their corrected union has affine dimension 50. In particular, the jump from 44 to 50 is a strong diagnostic that the second orientation was not accidentally generated as a copy of the first.


In [7]:

"""
    response(mask, u, v)

Evaluate one of the 16 deterministic binary functions of two binary inputs.
Bit `2u + v` of `mask` stores the output for the input pair `(u,v)`.
"""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""
    deterministic_vertex(fA, fB, fC, orientation)

Construct one deterministic 64-component behavior in canonical coordinates.

For `:clockwise`:
    a = fA(x,z),  b = fB(y,x),  c = fC(z,y).

For `:counterclockwise`:
    a = fA(x,y),  b = fB(y,z),  c = fC(z,x).
"""
function deterministic_vertex(
    fA::Int,
    fB::Int,
    fC::Int,
    orientation::Symbol,
)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

"""Generate the 4096 deterministic vertices of one cycle orientation."""
function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1

    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end

    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function affine_dimension(points; atol=1e-9)
    n = size(points, 1)
    n == 0 && return -1
    n == 1 && return 0

    differences = Float64.(points[2:end, :]) .- Float64.(points[1:1, :])
    return rank(differences; atol=atol, rtol=0.0)
end

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64

    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)

        for context in 0:7
            block = (8*context + 1):(8*context + 8)
            @assert sum(vertex[block]) == 1
        end
    end

    return true
end

vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)

# `unique(...; dims=1)` removes the 64 local deterministic behaviors that occur
# in both orientations. Keeping duplicates would not change the LP, but it would
# obscure the actual vertex count and the geometry.
vertices_cycle = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_cycle)

@assert size(vertices_clockwise, 1) == 4096
@assert size(vertices_counterclockwise, 1) == 4096
@assert length(intersect(clockwise_set, counterclockwise_set)) == 64
@assert size(vertices_cycle, 1) == 8128

@assert affine_dimension(vertices_clockwise) == 44
@assert affine_dimension(vertices_counterclockwise) == 44
@assert affine_dimension(vertices_cycle) == 50

println("Corrected cycle polytope")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  intersection:              ", length(intersect(clockwise_set, counterclockwise_set)))
println("  distinct union:            ", size(vertices_cycle, 1))
println("  affine dimension:          ", affine_dimension(vertices_cycle))




Corrected cycle polytope
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  intersection:              64
  distinct union:            8128
  affine dimension:          50


## 3. Published representatives of NS classes 27, 29, 43, and 44

The four targets are entered through the 26 correlators of a binary tripartite nonsignalling behavior. Their stored order is

$$
\begin{aligned}
&(A_0,A_1;\ B_0,B_1;\ C_0,C_1;\\
&AB_{00},AB_{01},AB_{10},AB_{11};\\
&AC_{00},AC_{01},AC_{10},AC_{11};\\
&BC_{00},BC_{01},BC_{10},BC_{11};\\
&ABC_{000},ABC_{001},ABC_{010},ABC_{011},
ABC_{100},ABC_{101},ABC_{110},ABC_{111}).
\end{aligned}
$$

These representatives are those reported in Table 1 of S. Pironio, J.-D. Bancal, and V. Scarani, *Extremal correlations of the tripartite no-signaling polytope*, J. Phys. A **44**, 065303 (2011), arXiv:1101.2477.

To convert correlators into probabilities, define $s_a=(-1)^a$, and analogously for $b$ and $c$. Then

$$
\begin{aligned}
p(abc\mid xyz)=\frac18[&1+s_aA_x+s_bB_y+s_cC_z
+s_as_bAB_{xy}+s_as_cAC_{xz}\\
&+s_bs_cBC_{yz}+s_as_bs_cABC_{xyz}].
\end{aligned}
$$

The implementation uses Julia rational numbers such as `1//3` throughout this conversion. Consequently, positivity, normalization, nonsignalling, and the later evaluations of rationalized inequalities can be checked exactly rather than up to floating-point tolerance.


In [9]:
#
# Correlator order:
#
#   A0,A1, B0,B1, C0,C1,
#   AB00,AB01,AB10,AB11,
#   AC00,AC01,AC10,AC11,
#   BC00,BC01,BC10,BC11,
#   ABC000,ABC001,ABC010,ABC011,ABC100,ABC101,ABC110,ABC111.
#
# These representatives agree with Table 1 of:
# S. Pironio, J.-D. Bancal, and V. Scarani,
# "Extremal correlations of the tripartite no-signaling polytope",
# J. Phys. A 44, 065303 (2011), arXiv:1101.2477.

const NS_CORRELATORS = Dict(
    27 => Rational{Int}[
        1//2, 0, 0, 0, 0, 0,
        1//2, 1//2, 0, 0,
        1//2, 1//2, 0, 0,
        0, 0, 0, 0,
        -1//2, -1//2, -1//2, -1//2, 1, 1, 1, -1,
    ],
    29 => Rational{Int}[
        1//3, 1//3, 1//3, 1//3, 1//3, 1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1//3, 1//3, 1//3, -1, 1//3, -1, -1, -1,
    ],
    43 => Rational{Int}[
        1//7, 1//7, 1//7, 1//7, 1//7, 1//7,
        3//7, -1//7, -1//7, -1//7,
        3//7, -1//7, -1//7, -1//7,
        -1//7, -1//7, -1//7, -1//7,
        1//7, 5//7, 5//7, -1, -1, -1, -1, -1,
    ],
    44 => Rational{Int}[
        0, 0, 0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, -1,
    ],
)

outcome_sign(bit::Int) = bit == 0 ? 1 : -1

"""Convert the 26 NS correlators to the canonical 64 probabilities exactly."""
function correlators_to_probabilities(correlators)
    @assert length(correlators) == 26
    q = Rational{Int}.(correlators)
    probabilities = fill(0//1, 64)

    A = q[1:2]
    B = q[3:4]
    C = q[5:6]
    AB = q[7:10]
    AC = q[11:14]
    BC = q[15:18]
    ABC = q[19:26]

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        sa = outcome_sign(a)
        sb = outcome_sign(b)
        sc = outcome_sign(c)

        probabilities[prob_index(a, b, c, x, y, z)] = (
            1 +
            sa*A[x + 1] +
            sb*B[y + 1] +
            sc*C[z + 1] +
            sa*sb*AB[2*x + y + 1] +
            sa*sc*AC[2*x + z + 1] +
            sb*sc*BC[2*y + z + 1] +
            sa*sb*sc*ABC[4*x + 2*y + z + 1]
        ) / 8
    end

    return probabilities
end

const NS_TARGETS = Dict(
    class => correlators_to_probabilities(NS_CORRELATORS[class])
    for class in CLASSES
)




Dict{Int64, Vector{Rational{Int64}}} with 4 entries:
  29 => [2//3, 0, 0, 0, 0, 0, 0, 1//3, 1//3, 1//3  …  0, 0, 0, 1//3, 1//3, 0, 1…
  43 => [2//7, 1//7, 1//7, 0, 0, 1//7, 1//7, 1//7, 2//7, 1//7  …  0, 1//7, 0, 2…
  27 => [1//4, 1//4, 1//4, 0, 0, 0, 0, 1//4, 1//4, 1//4  …  1//4, 0, 0, 1//4, 1…
  44 => [1//4, 0, 0, 1//4, 0, 1//4, 1//4, 0, 1//4, 0  …  1//4, 0, 0, 1//4, 1//4…

## 4. Exact target checks: positivity, normalization, nonsignalling, and extremality

Before using a target in an LP, this cell verifies the defining probabilistic and nonsignalling conditions directly in the 64-coordinate representation:

- every probability is nonnegative;
- the eight probabilities for each fixed input triple $(x,y,z)$ sum to 1;
- the $AB$ marginal is independent of Charlie's input $z$;
- the $AC$ marginal is independent of Bob's input $y$;
- the $BC$ marginal is independent of Alice's input $x$.

It then independently checks that each representative is a **vertex** of the nonsignalling polytope. The normalization and nonsignalling equalities are assembled as a (redundant) $56\times64$ matrix. Its rank is 38, hence the nonsignalling affine space has dimension

$$
64-38=26.
$$

At a given target, every zero probability saturates one positivity inequality. The target is extremal when the equality rows together with these active positivity rows have rank 64: there is then no nonzero feasible direction that keeps all active constraints saturated. The printed `active rank = 64 (NS vertex)` is therefore a geometric check, not merely a lookup of the class label.

All these tests use exact target probabilities. A failed assertion indicates an inconsistent representative, correlator order, probability order, or nonsignalling implementation and should be investigated before proceeding.


In [11]:

function validate_nonsignalling_behavior(probabilities)
    @assert length(probabilities) == 64
    @assert all(probability -> probability >= 0, probabilities)

    # Normalization for every input triple.
    for (x, y, z) in TRIPLES
        @assert sum(
            probabilities[prob_index(a, b, c, x, y, z)]
            for (a, b, c) in TRIPLES
        ) == 1
    end

    # AB marginal independent of z.
    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, y, 0)] for c in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, y, 1)] for c in 0:1)
        @assert lhs == rhs
    end

    # AC marginal independent of y.
    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, 0, z)] for b in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, 1, z)] for b in 0:1)
        @assert lhs == rhs
    end

    # BC marginal independent of x.
    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, 0, y, z)] for a in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, 1, y, z)] for a in 0:1)
        @assert lhs == rhs
    end

    return true
end

"""
Return a (redundant) matrix for normalization and tripartite NS equalities.
Its rank must be 38, leaving the familiar 26-dimensional NS affine space.
"""
function nonsignalling_equality_matrix()
    equalities = zeros(Float64, 56, 64)
    row = 0

    for (x, y, z) in TRIPLES
        row += 1
        for (a, b, c) in TRIPLES
            equalities[row, prob_index(a, b, c, x, y, z)] = 1
        end
    end

    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        row += 1
        for c in 0:1
            equalities[row, prob_index(a, b, c, x, y, 0)] += 1
            equalities[row, prob_index(a, b, c, x, y, 1)] -= 1
        end
    end

    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        row += 1
        for b in 0:1
            equalities[row, prob_index(a, b, c, x, 0, z)] += 1
            equalities[row, prob_index(a, b, c, x, 1, z)] -= 1
        end
    end

    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        row += 1
        for a in 0:1
            equalities[row, prob_index(a, b, c, 0, y, z)] += 1
            equalities[row, prob_index(a, b, c, 1, y, z)] -= 1
        end
    end

    @assert row == 56
    return equalities
end

const NS_EQUALITIES = nonsignalling_equality_matrix()
@assert rank(NS_EQUALITIES; atol=1e-9, rtol=0.0) == 38

"""
Check extremality in the NS polytope using the rank of the NS equalities plus
the positivity inequalities saturated by the target.
"""
function ns_active_rank(probabilities)
    zero_coordinates = findall(iszero, probabilities)
    coordinate_rows = Matrix{Float64}(I, 64, 64)[zero_coordinates, :]
    active_matrix = vcat(NS_EQUALITIES, coordinate_rows)
    return rank(active_matrix; atol=1e-9, rtol=0.0)
end

println("\nTarget checks")
for class in CLASSES
    target = NS_TARGETS[class]
    validate_nonsignalling_behavior(target)
    active_rank = ns_active_rank(target)
    @assert active_rank == 64

    @printf(
        "  class %2d: support = %2d, active rank = %2d (NS vertex)\n",
        class,
        count(value -> !iszero(value), target),
        active_rank,
    )
end





Target checks
  class 27: support = 32, active rank = 64 (NS vertex)
  class 29: support = 23, active rank = 64 (NS vertex)
  class 43: support = 36, active rank = 64 (NS vertex)
  class 44: support = 32, active rank = 64 (NS vertex)


## 5. White-noise-normalized separation LP

For an inequality $c\cdot p\geq\beta$, define its signed slack at $q$ by

$$
s(q)=c\cdot q-\beta.
$$

The LP imposes $s(v)\geq0$ at every siglocal vertex and $s(u)=1$ at the
uniform behavior. It then minimizes

$$
\delta=s(p_{\mathrm{NS}}).
$$

If $\delta<0$, the target violates the inequality. For the noisy behavior

$$
p_r=(1-r)p_{\mathrm{NS}}+ru,
$$

the boundary is reached at

$$
r_{\mathrm{crit}}=\frac{-\delta}{1-\delta}.
$$

Equivalently, its critical visibility is

$$
\eta_{\mathrm{crit}}=1-r_{\mathrm{crit}}
=\frac{1}{1-\delta}.
$$

Because $r_{\mathrm{crit}}$ increases as $\delta$ becomes more negative, this
LP directly maximizes white-noise resistance of the selected NS target.

The uniform behavior is not inserted by assumption alone: the code verifies
that it is exactly the average of the 8128 distinct cycle vertices.


In [13]:
const UNIFORM_BEHAVIOUR = fill(1//8, 64)

# Every coordinate occurs in exactly 1016 of the 8128 vertices. Hence their
# uniform average is precisely u(abc|xyz)=1/8.
@assert all(sum(vertices_cycle; dims=1) .== 1016)
@assert [
    sum(Int(vertices_cycle[i, j]) for i in axes(vertices_cycle, 1)) //
    size(vertices_cycle, 1)
    for j in axes(vertices_cycle, 2)
] == UNIFORM_BEHAVIOUR

"""Rationalize one numerical LP value, treating numerical zeros explicitly."""
function rationalize_scalar(value; tol=1e-8)
    return abs(value) <= tol ? 0//1 : rationalize(Int, value; tol=tol)
end

"""Convert a rational inequality to a primitive integer representative."""
function primitive_integer_form(coefficients, beta)
    all_values = vcat(coefficients, [beta])
    common_denominator = foldl(lcm, denominator.(all_values); init=1)
    integer_values = [
        numerator(value * common_denominator)
        for value in all_values
    ]

    common_divisor = foldl(gcd, abs.(integer_values); init=0)
    common_divisor == 0 && error("The reconstructed inequality is zero")
    integer_values .÷= common_divisor

    return integer_values[1:end-1], integer_values[end]
end

"""
    solve_white_noise_separator(class, target, ...)

Solve the free-coefficient LP, reconstruct its rational solution, and audit
the resulting inequality exactly on all 8128 vertices.
"""
function solve_white_noise_separator(
    class::Int,
    target,
    vertices_cycle,
    vertices_clockwise,
    vertices_counterclockwise;
    tol=1e-8,
    verbose=false,
)
    number_of_vertices, dimension = size(vertices_cycle)
    @assert dimension == length(target) == length(UNIFORM_BEHAVIOUR) == 64

    model = Model(GLPK.Optimizer)
    verbose || set_silent(model)

    # No sign or box constraint is imposed: every coefficient and beta is free.
    @variable(model, coefficients[1:dimension])
    @variable(model, beta)

    @constraint(
        model,
        valid_on_cycle[i=1:number_of_vertices],
        sum(vertices_cycle[i, j] * coefficients[j] for j in 1:dimension) >= beta,
    )

    @constraint(
        model,
        white_noise_normalization,
        sum(
            Float64(UNIFORM_BEHAVIOUR[j]) * coefficients[j]
            for j in 1:dimension
        ) - beta == 1,
    )

    @objective(
        model,
        Min,
        sum(Float64(target[j]) * coefficients[j] for j in 1:dimension) - beta,
    )

    optimize!(model)

    status = termination_status(model)
    status == MOI.OPTIMAL || error(
        "White-noise LP for class $class ended with status $status",
    )

    coefficients_float = value.(coefficients)
    beta_float = value(beta)
    coefficients_exact = [
        rationalize_scalar(value; tol=tol)
        for value in coefficients_float
    ]
    beta_exact = rationalize_scalar(beta_float; tol=tol)

    exact_slack(vertex) = sum(
        coefficients_exact[j] * Int(vertex[j])
        for j in 1:dimension
    ) - beta_exact

    exact_cycle_slacks = [
        exact_slack(@view vertices_cycle[i, :])
        for i in axes(vertices_cycle, 1)
    ]
    exact_clockwise_slacks = [
        exact_slack(@view vertices_clockwise[i, :])
        for i in axes(vertices_clockwise, 1)
    ]
    exact_counterclockwise_slacks = [
        exact_slack(@view vertices_counterclockwise[i, :])
        for i in axes(vertices_counterclockwise, 1)
    ]

    exact_uniform_slack = sum(
        coefficients_exact[j] * UNIFORM_BEHAVIOUR[j]
        for j in 1:dimension
    ) - beta_exact
    exact_target_slack = sum(
        coefficients_exact[j] * target[j]
        for j in 1:dimension
    ) - beta_exact

    minimum(exact_cycle_slacks) >= 0 || error(
        "Rational reconstruction produced an invalid inequality for class $class",
    )
    exact_uniform_slack == 1 || error(
        "Rational reconstruction changed the white-noise normalization for class $class",
    )
    exact_target_slack < 0 || error(
        "The reconstructed class-$class inequality does not separate its target",
    )
    abs(Float64(exact_target_slack) - objective_value(model)) <= 1e-7 || error(
        "Exact and numerical target slacks disagree for class $class",
    )

    saturating_indices = findall(iszero, exact_cycle_slacks)
    face_dimension = affine_dimension(vertices_cycle[saturating_indices, :])
    polytope_dimension = affine_dimension(vertices_cycle)

    critical_visibility = 1 // 1 / (1 - exact_target_slack)
    critical_noise_fraction = -exact_target_slack / (1 - exact_target_slack)

    primitive_coefficients, primitive_beta = primitive_integer_form(
        coefficients_exact,
        beta_exact,
    )

    return (
        class=class,
        status=status,
        coefficients=coefficients_float,
        beta=beta_float,
        exact_coefficients=coefficients_exact,
        exact_beta=beta_exact,
        solver_target_slack=objective_value(model),
        exact_target_slack=exact_target_slack,
        exact_uniform_slack=exact_uniform_slack,
        exact_cycle_minimum_slack=minimum(exact_cycle_slacks),
        exact_clockwise_minimum_slack=minimum(exact_clockwise_slacks),
        exact_counterclockwise_minimum_slack=minimum(exact_counterclockwise_slacks),
        critical_visibility=critical_visibility,
        critical_noise_fraction=critical_noise_fraction,
        saturation_count=length(saturating_indices),
        face_dimension=face_dimension,
        is_facet=(face_dimension == polytope_dimension - 1),
        primitive_coefficients=primitive_coefficients,
        primitive_beta=primitive_beta,
    )
end

separator_results = Dict{Int, Any}()

for class in CLASSES
    separator_results[class] = solve_white_noise_separator(
        class,
        NS_TARGETS[class],
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise,
    )
end

# Independently obtained optimal signed slacks. These assertions protect
# against a future coordinate-order, target, vertex, or LP change.
const EXPECTED_TARGET_SLACKS = Dict(
    27 => -1//11,
    29 => -8//45,
    43 => -2//21,
    44 => -1//3,
)

for class in CLASSES
    @assert separator_results[class].exact_target_slack ==
            EXPECTED_TARGET_SLACKS[class]
end

println("\nWhite-noise-normalized separation results")
println("class  target-slack  visibility  noise-frac  saturating  face-dim  facet")

for class in CLASSES
    result = separator_results[class]
    @printf(
        "%5d  %12s  %10s  %10s  %10d  %8d  %s\n",
        class,
        string(result.exact_target_slack),
        string(result.critical_visibility),
        string(result.critical_noise_fraction),
        result.saturation_count,
        result.face_dimension,
        string(result.is_facet),
    )
end



White-noise-normalized separation results
class  target-slack  visibility  noise-frac  saturating  face-dim  facet
   27        -1//11      11//12       1//12         119        49  true
   29        -8//45      45//53       8//53         270        49  true
   43        -2//21      21//23       2//23         508        49  true
   44         -1//3        3//4        1//4         508        49  true


## 6. Exact inequality matrices

For each class, the notebook displays two equivalent forms:

1. the rational LP normalization $c\cdot u-\beta=1$;
2. a primitive integer rescaling, useful for presentation and subsequent NPA
   calculations.

Rows label $xyz$ and columns label $abc$, both in the order

`000, 001, 010, 011, 100, 101, 110, 111`.

The facet label is determined afterward from the affine dimension of the
saturating siglocal vertices. Facetness is not assumed by the LP.


In [15]:
coefficient_matrix(coefficients) =
    Matrix(permutedims(reshape(coefficients, 8, 8)))

normalized_inequality_matrices = Dict(
    class => coefficient_matrix(separator_results[class].exact_coefficients)
    for class in CLASSES
)

primitive_inequality_matrices = Dict(
    class => coefficient_matrix(separator_results[class].primitive_coefficients)
    for class in CLASSES
)

println("\nExact coefficient matrices")
for class in CLASSES
    result = separator_results[class]

    println("\n", repeat("=", 72))
    println("Class $class")
    println("Normalized inequality: c . p >= ", result.exact_beta)
    println("Normalization check:  c . u - beta = ", result.exact_uniform_slack)
    display(normalized_inequality_matrices[class])

    println("Primitive integer inequality: C . p >= ", result.primitive_beta)
    display(primitive_inequality_matrices[class])
end



Exact coefficient matrices

Class 27
Normalized inequality: c . p >= -1//1
Normalization check:  c . u - beta = 1//1


8×8 Matrix{Rational{Int64}}:
 -2//11  -1//11  -3//11    0     -1//11   2//11    0     -3//11
 -5//11  -3//11  -4//11    0     -4//11    0     -1//11  -3//11
  1//11    0       0      1//11   2//11   3//11   3//11  -2//11
 -1//11  -3//11    0       0       0       0      3//11  -3//11
   0      3//11   5//11    0      1//11  -2//11    0      3//11
   0      2//11   3//11  -3//11   3//11  -1//11    0      2//11
 -2//11   3//11   3//11    0      1//11    0       0      5//11
   0       0     -3//11   3//11  -3//11   3//11    0     -2//11

Primitive integer inequality: C . p >= -11


8×8 Matrix{Int64}:
 -2  -1  -3   0  -1   2   0  -3
 -5  -3  -4   0  -4   0  -1  -3
  1   0   0   1   2   3   3  -2
 -1  -3   0   0   0   0   3  -3
  0   3   5   0   1  -2   0   3
  0   2   3  -3   3  -1   0   2
 -2   3   3   0   1   0   0   5
  0   0  -3   3  -3   3   0  -2


Class 29
Normalized inequality: c . p >= -32//15
Normalization check:  c . u - beta = 1//1


...

8×8 Matrix{Int64}:
  0   0   0  -1   1   0   0   0
 -1   0   0   0  -1  -1  -1   0
  0   1   0   0   0   0  -1   0
  0  -1   0   0   0   0   1   0
 -1  -1  -1   0  -1   0   0   0
  1   0   0   0   0   0   0  -1
  1   0   1   1   0   0   1   0
  0   0  -1   0   0   1   0   0


Class 44
Normalized inequality: c . p >= 1//3
Normalization check:  c . u - beta = 1//1


8×8 Matrix{Rational{Int64}}:
  0    1//3  1//3   0    1//3   0     0    1//3
  0    1//3  1//3   0    1//3   0     0    1//3
  0    1//3  1//3   0    1//3   0     0    1//3
  0    1//3  1//3   0    1//3   0     0    1//3
  0    1//3  1//3   0    1//3   0     0    1//3
  0    1//3  1//3   0    1//3   0     0    1//3
  0    1//3  1//3   0    1//3   0     0    1//3
 1//3   0     0    1//3   0    1//3  1//3   0

Primitive integer inequality: C . p >= 1

8×8 Matrix{Int64}:
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 1  0  0  1  0  1  1  0

## 7. Save machine-readable results

The summary file records the optimized slack, visibility, white-noise fraction,
and face data. Two coefficient files store the normalized rational and
primitive integer inequalities in canonical 64-coordinate order.


In [17]:
RESULTS_DIRECTORY = joinpath(pwd(), "cycle_white_noise_lp_results")
mkpath(RESULTS_DIRECTORY)

summary_path = joinpath(RESULTS_DIRECTORY, "white_noise_separator_summary.tsv")
normalized_path = joinpath(RESULTS_DIRECTORY, "normalized_coefficients.tsv")
primitive_path = joinpath(RESULTS_DIRECTORY, "primitive_integer_coefficients.tsv")

open(summary_path, "w") do io
    println(io, join([
        "class",
        "target_slack",
        "critical_visibility",
        "critical_noise_fraction",
        "normalized_beta",
        "saturation_count",
        "face_dimension",
        "is_facet",
    ], "\t"))

    for class in CLASSES
        result = separator_results[class]
        println(io, join([
            string(class),
            string(result.exact_target_slack),
            string(result.critical_visibility),
            string(result.critical_noise_fraction),
            string(result.exact_beta),
            string(result.saturation_count),
            string(result.face_dimension),
            string(result.is_facet),
        ], "\t"))
    end
end

coordinate_headers = ["c_$index" for index in 1:64]

open(normalized_path, "w") do io
    println(io, join(vcat(["class", "beta"], coordinate_headers), "\t"))
    for class in CLASSES
        result = separator_results[class]
        println(io, join(vcat(
            [string(class), string(result.exact_beta)],
            string.(result.exact_coefficients),
        ), "\t"))
    end
end

open(primitive_path, "w") do io
    println(io, join(vcat(["class", "beta"], coordinate_headers), "\t"))
    for class in CLASSES
        result = separator_results[class]
        println(io, join(vcat(
            [string(class), string(result.primitive_beta)],
            string.(result.primitive_coefficients),
        ), "\t"))
    end
end

println("\nSaved results")
println("  ", abspath(summary_path))
println("  ", abspath(normalized_path))
println("  ", abspath(primitive_path))



Saved results
  C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_white_noise_lp_results\white_noise_separator_summary.tsv
  C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_white_noise_lp_results\normalized_coefficients.tsv
  C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_white_noise_lp_results\primitive_integer_coefficients.tsv


## Interpretation of the output

For every class:

- a negative `target_slack` confirms separation from the siglocal polytope;
- `critical_noise_fraction` is the fraction of uniform white noise required to
  reach the boundary detected by the optimal inequality;
- `is_facet` is established from the saturating vertices after the LP;
- the exact validity audit requires minimum siglocal slack $0$ and uniform
  slack $1$.

This optimization is tailored to the selected NS extremal and white-noise
direction. It does **not** directly optimize quantum violation. Any resulting
facet must therefore be tested separately with NPA, exactly as in the previous
pipeline.
